# Week 4 — Monday: Reshaping Data — Melting and Pivoting

**DATA 202 · Calvin University**

Two questions we will ask over and over today — before *every* line of reshaping code:

> 🧭 **What is each row about?**
> 🧭 **What do we want each row to be about?**

Reshaping is just the move from the first answer to the second.

**Today's plan (50 min):**

| Time | Part |
|---|---|
| ~20 min | **Part 1 — Demo (together):** garden harvests — melt, pivot, explode |
| ~25 min | **Part 2 — Your turn:** bike-share stations — same three moves, completely different data |
| ~5 min | Careful with reshaping + what's next |

---
## Part 1 — Demo: Garden Harvests · ~20 min

A community garden logs the weight of each plot's harvest, week by week.

In [ ]:
import pandas as pd

DATA_PATH = "https://cs.calvin.edu/courses/data/202/26fa/datasets/plot_yields.csv"
yields = pd.read_csv(DATA_PATH)
yields.head()

In [ ]:
yields.info()

- 18 rows — one per garden plot
- 6 columns (`Week1_lbs`...`Week6_lbs`) — one week's harvest each, side by side
- `Crops` crams several values into one cell (`"lettuce, squash"`)

> 🧭 **Row check**
> - *What is each row about?* → **one plot's whole season**
> - *What do we want each row to be about?* → **depends on our question — say we want to compare weeks…**

> 💬 **Question:** We want the total harvest for Week 3 across all 18 plots. Can one `.groupby()` do that right now?

<details><summary>Answer</summary>

No — there is nothing to group *by*. "Week" isn't a value in any column; it's hiding in the column *names*.
</details>

### Wide vs. Long

![A wide table with one row per plot and separate Week1/Week2/Week3 columns, next to a long table with one row per plot-per-week and a single Week column plus a single Harvest_lbs column. Arrows labeled "melt" and "pivot" connect the two, with the caption: one row = one plot's whole season (wide) vs. one row = one plot, in one week (long).](https://cs.calvin.edu/courses/data/202/26fa/weeks/04/images/wide_long_diagram.png)

**Same 6 numbers in both tables** — nothing added, nothing lost. Only the meaning of a row changed.

* **Wide:** one row = **one plot's whole season** → G01's Week 2 harvest is row `G01`, column `Week2_lbs`.
* **Long:** one row = **one plot, in one week** → the same number is the row where `Plot_ID`=`G01` *and* `Week`=`Week2`, column `Harvest_lbs`.

**`melt()`**: column *names* → values in a new column; their contents → values in another new column.
**`pivot_table()`**: the reverse.

> 💬 **Question:** Which shape would you want for *"every week's harvest for plot G01, side by side"*? Which for *"average harvest per plot-week, across the garden"*?

<details><summary>Answer</summary>

Wide for the first, long for the second. Neither is "correct" — each is built for a different question.
</details>

### Melting: wide → long

> 🧭 **Row check**
> - *What is each row about?* → **one plot's whole season**
> - *What do we want each row to be about?* → **one plot, in one week**

> 💬 **Question:** If we melt the six `Week*_lbs` columns, how many rows will the new table have?

<details><summary>Answer</summary>

18 plots × 6 weeks = **108** rows. Melting multiplies the row count by the number of columns unstacked.
</details>

In [ ]:
long = pd.melt(
    yields,
    id_vars=["Plot_ID", "Gardener", "Crops"],
    value_vars=["Week1_lbs", "Week2_lbs", "Week3_lbs", "Week4_lbs", "Week5_lbs", "Week6_lbs"],
    var_name="Week",
    value_name="Harvest_lbs",
)
long["Week"] = long["Week"].str.replace("_lbs", "", regex=False)
long.shape

In [ ]:
long[long['Plot_ID'] == 'G01']

- `id_vars` → columns **repeated** on every row (they still identify the plot)
- `value_vars` → columns **unstacked**: their *names* become `Week` values, their *contents* become `Harvest_lbs` values

> 🧭 **Row check**
> - *What is each row about?* → **one plot, in one week**
> - *What do we want each row to be about?* → **one *week* (total over all plots)**

Now "week" is a value in a column, so `.groupby()` has something to group by:

In [ ]:
long.groupby('Week', sort=False)['Harvest_lbs'].sum().round(1)

The season peaks in Week 3 (194.8 lbs) and tapers by Week 6 (67.5 lbs).

> 💬 **Question:** How would you find the single best plot-week of the whole season in the *wide* table? And in the *long* one?

<details><summary>Answer</summary>

Wide: run `idxmax()` on each of six columns and compare the results by hand. Long: one `idxmax()` on `Harvest_lbs` — because every row is already "one plot in one week".
</details>

In [ ]:
best = long.loc[long["Harvest_lbs"].idxmax()]
best[["Plot_ID", "Week", "Harvest_lbs"]]

### Pivoting: long → wide

> 🧭 **Row check**
> - *What is each row about?* → **one plot, in one week**
> - *What do we want each row to be about?* → **one plot's whole season (again)**

> 💬 **Question:** If we pivot `long` back with `Week` as the columns, will the result match `yields` exactly? Predict the shape.

<details><summary>Answer</summary>

Same information, so the same shape: **(18, 9)** — 3 identifying columns + 6 week columns. Melt and pivot are inverses.
</details>

In [ ]:
back_to_wide = long.pivot_table(
    index=["Plot_ID", "Gardener", "Crops"],
    columns="Week",
    values="Harvest_lbs",
).reset_index()
back_to_wide.shape

- `index` → columns that identify a row
- `columns` → the column whose *values* become new headers
- `values` → what fills the cells

*If a `Plot_ID` + `Week` pair appeared twice, `pivot_table` would have to squash the duplicates into one cell — by default with `aggfunc="mean"`. No duplicates here, but that's the safety net.*

### Exploding: one cell, many values

`Crops` holds several values per cell (`"lettuce, squash"`), so "how many plots grow lettuce?" can't be answered directly.

> 🧭 **Row check**
> - *What is each row about?* → **one plot**
> - *What do we want each row to be about?* → **one plot **growing one crop****

Split the text into a list, then `.explode()` gives each crop its own row:

In [ ]:
yields_crops = yields.copy()
yields_crops["Crop_List"] = yields_crops["Crops"].str.split(", ")
exploded = yields_crops.explode("Crop_List").reset_index(drop=True)
exploded.shape

In [ ]:
exploded['Crop_List'].value_counts()

Lettuce and kale tie at 7 of 18 plots.

> 💬 **Question:** If we now sum `Week3_lbs` on `exploded`, do we still get Week 3's total harvest?

<details><summary>Answer</summary>

No! A plot growing two crops now appears in **two rows**, so its harvest is counted twice. `.explode()` multiplies rows, not harvest weight. Run the cell below to see it.
</details>

In [ ]:
print("original yields :", yields["Week3_lbs"].sum().round(1))
print("after explode   :", exploded["Week3_lbs"].sum().round(1))

---
## Part 2 — Your Turn: Bike-Share Stations · ~25 min

A city bike-share system counts the rides that start at each of its 12 stations on each day of the week. Same three moves as the garden — **melt, pivot, explode** — but a completely different dataset.

### How to work every task

1. **Ask the row questions** — each task starts with a 🧭 *Row check*. Say the answers out loud before you type.
2. **Pick the tool** that moves you from the first answer to the second:

| Row now… | …you want | Tool |
|---|---|---|
| several columns holding the same kind of value (`Mon_rides`, `Tue_rides`, …) | one row per value | `pd.melt()` |
| one row per (thing, category) | one row per thing, categories as columns | `.pivot_table()` |
| several values crammed in one cell (`"cafe, park"`) | one row per value | `.str.split()` then `.explode()` |
| rows you want to collapse into totals | one row per group | `.groupby()` |

3. **Check the shape** (`.shape`) against the *Check yourself* answer before moving on — a wrong shape means the row meaning is off.

Stuck? Open the **Hint** — it gives the skeleton with blanks (`___`) to fill in.

In [ ]:
bikes = pd.read_csv("https://cs.calvin.edu/courses/data/202/26fa/datasets/bike_stations.csv")
bikes.head()

The days of the week are hiding in the column names, and `Nearby` lists several places in one cell.

---
### 🔨 Task 1 — Melt · ~5 min

> 🧭 **Row check**
> - *What is each row about?* → **one station's whole week**
> - *What do we want each row to be about?* → **one station, on one day**

Melt the seven `*_rides` columns into `rides_long`. New columns: `Day` and `Rides`. Strip `_rides` from the day names (as in the demo).

<details><summary>Hint</summary>

```python
rides_long = pd.melt(
    bikes,
    id_vars=[___],          # columns to repeat on every row
    value_vars=[___],       # the seven *_rides columns
    var_name=___,
    value_name=___,
)
rides_long["Day"] = rides_long["Day"].str.replace(___, "", regex=False)
```
Tip: `[c for c in bikes.columns if c.endswith("_rides")]` builds the `value_vars` list for you.
</details>

<details><summary>Check yourself</summary>

`rides_long.shape` is `(84, 5)` — 12 stations × 7 days.
</details>

In [ ]:
# Your code here
rides_long = None

---
### 🔨 Task 2 — Busiest Station-Day · ~4 min

> 🧭 **Row check**
> - *What is each row about?* → **one station, on one day**
> - *What do we want each row to be about?* → **one station, on one day  (already right!)**

No reshaping needed — the long table already has the right rows. Find the single highest `Rides` value: which station, which day, how many rides. Store them in `best_station`, `best_day`, `best_rides`.

<details><summary>Hint</summary>

Same two steps as the demo: `idxmax()` on the `Rides` column gives the row's index; `.loc[that_index]` gives the whole row. Then read `Station`, `Day`, `Rides` off that row.
</details>

<details><summary>Check yourself</summary>

Memorial Hospital, Thursday, 95 rides.
</details>

In [ ]:
# Your code here

---
### 🔨 Task 3 — Rides per Day · ~3 min

> 🧭 **Row check**
> - *What is each row about?* → **one station, on one day**
> - *What do we want each row to be about?* → **one *day* (all stations added up)**

Total the rides for each day across all stations. Which day is busiest? Which is quietest?

<details><summary>Hint</summary>

```python
rides_long.groupby(___, sort=False)[___].sum()
```
`sort=False` keeps the days in Mon → Sun order.
</details>

<details><summary>Check yourself</summary>

Mon 534 · Tue 561 · Wed 549 · Thu 552 · Fri 566 · Sat 550 · Sun 454 — busiest Friday, quietest Sunday.
</details>

In [ ]:
# Your code here

---
### 🔨 Task 4 — Pivot Back · ~4 min

> 🧭 **Row check**
> - *What is each row about?* → **one station, on one day**
> - *What do we want each row to be about?* → **one station's whole week (again)**

Pivot `rides_long` back to wide, then `.reset_index()`. Store it in `back_to_wide`.

<details><summary>Hint</summary>

```python
back_to_wide = rides_long.pivot_table(
    index=[___],      # the columns that identify a station
    columns=___,      # the column whose values become headers
    values=___,       # what fills the cells
).reset_index()
```
</details>

<details><summary>Check yourself</summary>

Shape `(12, 10)`. The day columns come out in **alphabetical** order (Fri, Mon, Sat, …) — `pivot_table` sorts new headers. The garden's `Week1`…`Week6` happened to sort correctly; days of the week don't.
</details>

In [ ]:
# Your code here
back_to_wide = None

---
### 🔨 Task 5 — Explode the Amenities · ~5 min

> 🧭 **Row check**
> - *What is each row about?* → **one station**
> - *What do we want each row to be about?* → **one station **next to one amenity****

Which amenity is near the most stations? Store the count for the top amenity in `top_amenity_count` and the number of distinct amenities in `n_distinct_amenities`.

<details><summary>Hint</summary>

1. Copy `bikes`, then make a list column: `bikes["Amenity_List"] = bikes["Nearby"].str.split(___)` (what separates the values?)
2. `.explode("Amenity_List")` — one row per station-amenity pair
3. `.value_counts()` on `Amenity_List` for the counts; `.nunique()` for the number of distinct amenities
</details>

<details><summary>Check yourself</summary>

`cafe` is near 6 stations — no tie. There are 15 distinct amenities.
</details>

In [ ]:
# Your code here

---
### ⭐ Stretch (if you finish early) — Which Stations Are Weekend Stations?

> 🧭 **Row check**
> - *What is each row about?* → **one station, on one day**
> - *What do we want each row to be about?* → **one station, split into weekday vs. weekend**

Which stations average **more rides on weekends (Sat, Sun) than on weekdays**?

1. In `rides_long`, add a `Weekend` column: `rides_long["Day"].isin(["Sat", "Sun"])`
2. `pivot_table(index="Station", columns="Weekend", values="Rides", aggfunc="mean")`

<details><summary>Check yourself</summary>

Stadium Gate, Farmers Market, Riverside Park and Lakefront Pier are far busier on weekends; Memorial Hospital and Union Station drop the most.
</details>

In [ ]:
# Your code here

---
## Careful with Reshaping · ~5 min

Every reshape today was the same move — answer the two row questions, then let the right tool get you from one to the other:

> 🧭 **What is each row about?** → 🧭 **What do we want each row to be about?**

No information is added or lost — but the row's meaning changes what a careless `.sum()`/`.mean()` silently computes:

* **wide**: summing `Week3_lbs` = Week 3's total across every plot — one cell per plot
* **long**: summing all of `Harvest_lbs` = the season total — but only correct because every plot contributed exactly 6 rows. Skipped weeks (vs. zero-harvest weeks) would quietly bias any "average per plot-week" toward whoever reported more often.
* **exploded**: summing a per-plot number double-counts every plot with more than one crop.

**A table's shape is not neutral.** Wide makes "compare days side by side" trivial, long makes "group by day" trivial. Neither is *the* correct shape — only the one that fits the question.

**Still unanswerable either way:** which of these 18 plots are registered with the garden coordinator? That needs a *second* table — a new question about what connects one table's rows to another's. → Wednesday, with a materials lab and five tables.

---
## Coming Up

| Day | Topic | Builds on today |
|---|---|---|
| Wed | Joining tables — keys, primary/foreign keys, four join types | New data (a biomaterials lab, five tables) — "what is a row about," asked of *several* tables at once, and whether they should be joined at all |
| Week 5 | Clustering & Dimensionality Reduction | Finding groups the data suggests, instead of ones we choose (`Plot_ID`, `Week`) in advance |